<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [ ]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122


In [ ]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch
import os

# Params
VERBOSE = True
CHOSEN = 'llama'
FILE_NAMES = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
IT = 10
BASE_URL = 'https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'
BASE_FOLDER = 'drive/MyDrive/NLP_proj/estimation/'
OVERWRITE = ['']
MAX_RETRY = 10

# Load the model
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf",
              'temperature': 0.7,
              'n_ctx': 32768,
              'chat_format': "llama-3"
              },
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf",
             'temperature': 0.6,
             'n_ctx': 40960,
             'chat_format': "qwen"},
    'mistral': {'repo_id':"TheBloke/Mistral-7B-v0.1-GGUF",
                'filename':"mistral-7b-v0.1.Q8_0.gguf",
                'temperature': 0.7,
                'n_ctx': 32768,
                'chat_format': "chatml"},
}

model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=models[CHOSEN]['n_ctx'], # context size
                            flash_attn=True, # use flash attention
                            chat_format=models[CHOSEN]['chat_format'], # chat format
                            verbose=False,
                            force_download=True,
                            enable_thinking=True)


def generate_message(sys_prompt, usr_prompt):
    return [
        {
                "role": "system",
                "content": sys_prompt,
            },
            {
                "role": "user",
                "content": usr_prompt,
            },
    ]



prompts = {}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [34]:
import re
test1 = 'mechanics: [A, B, ..., Z]'
test2 = 'a\nbhst\nOverall Complexity: 4.2'
test3 ='tsr\noptimal number of pLayers: 10'
test4 ='tsr\noptimal number of Player: 1'
test_dur_1 = 'sthser\nduration: 10-20'
test_dur_2 = 'sthser\nduration: 120'

regex = {'mechanics': r"(?i)^.*\bmechanics\s*:\s*\[.*\]\s*$",
         'complexity': r'(?i)^.*\bcomplexity:\s*[1-5]\.\d\s*$',
         'player': r'(?i)^.*\bplayer.*:\s*\d+\s*$',
         'duration': r'(?i)^.*\bduration.*:\s*(\d+)(?:-(\d+))?'

}
def check_output(prompt, output):
    if(prompt == 'all'):
        try:
            json.loads(output)
            return True
        except:
            return False
    else:
        last_line = output.split('\n')[-1]
        return bool(re.match(regex[prompt], last_line))


print(check_output('mechanics',test1))
print(check_output('complexity',test2))
print(check_output('player',test3))
print(check_output('player',test4))
print(check_output('duration',test_dur_1))
print(check_output('duration',test_dur_2))


True
True
True
True
True
True


# Game parameter estimation
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

## Estimating everything at once

In [ ]:

prompts['all'] = """You are a board‑game analyst that always explains its reasoning before answering.
For any supplied rule excerpt you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting already existing BGG mechanic(s).
3. Judge the rule density and decision depth. Then assign a complexity score (1.0‑5.0).
4. From the number of components in the box and player‑interaction patterns infer the optimal player‑count.
6. Assess the progression of a typical game turn. Based on the complexity of the required actions and how much each turn brings the player closer to the final goal, estimate the game’s average duration in minutes.
Explain your reasoning step by step then output a json object with the fields 'mechanics' (list of strings), 'complexity'(1-5), 'optimal player count', 'duration' that matches the schema below.

<<<JSON schema>>>
{
  "type": "object",
  "properties": {
    "reasoning": {"type": "string"},
    "answer": {"type": "object", "properties": {
      "mechanics": {"type": "array", "items": {"type": "string"}},
      "complexity": {"type": "number", "minimum": 1.0, "maximum": 5.0},
      "optimal player count": {"type": "number"},
      "duration": {"type": "number"},
      "required": ["mechanics", "complexity", "optimal player count", "duration"]
    }}
  },
  "required": ["reasoning", "answer"]
}


<<<EXAMPLE>>>
{
  "reasoning": "The rules describe moving pieces on a grid, controlling regions, and a simple scoring system. These map to Area Control and Hand Management. The rule density is low and decisions are straightforward, so complexity is around 1.8. With only 30 tokens and a small board, the game works best with 2‑3 players; 2 is optimal for maximum interaction. Each turn shifts control of a few squares, and a full game finishes in roughly 30 minutes.",
  "answer": {
    "mechanics": ["Area Control", "Hand Management"],
    "complexity": 1.8,
    "optimal player count": 2,
    "duration": 30
  }
}
"""

all_rf = {"type": "json_object",
          "schema": {
              "type": "object",
              "properties": {
                  "reasoning": {"type": "string"},
                  "answer": {"type": "object", "properties": {
                      "mechanics": {"type": "array", "items": {"type": "string"}},
                      "complexity": {"type": "number", "minimum": 1, "maximum": 5},
                      "optimal player count": {"type": "number"},
                      "duration": {"type": "number"},
                      "required": ["mechanics", "complexity", "optimal player count", "duration"]
                      }
                    }
                  },
              "required": ["reasoning", "answer"]
              }
          }



## Estimating each parameter separately

### Mechanics

In [ ]:
prompts['mechanics'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s). Use only mechanics present in the list below.

here is a complete list of the available BGG mechanics:
[Acting, Action / Event, Action Drafting, Action Points, Action Queue, Action Retrieval,
Action Timer, Advantage Token, Alliances, Area Majority / Influence, Area Movement, Area-Impulse,
Auction / Bidding, Auction Compensation, Auction: Dexterity, Auction: Dutch, Auction: Dutch Priority,
Auction: English, Auction: Fixed Placement, Auction: Multiple Lot, Auction: Once Around,
Auction: Sealed Bid,Auction: Turn Order Until Pass, Automatic Resource Growth, Betting and Bluffing,
Bias, Bids As Wagers, Bingo, Bribery, Campaign / Battle Card Driven, Card Play Conflict Resolution,
Catch the Leader, Chaining, Chit-Pull System, Closed Drafting, Closed Economy Auction, Command Cards,
Commodity Speculation, Communication Limits, Connections, Constrained Bidding, Contracts,
Cooperative Game, Crayon Rail System, Critical Hits and Failures, Cube Tower, Deck Construction,
"Deck, Bag, and Pool Building", Deduction,Delayed Purchase, Dice Rolling, Die Icon Resolution,
Different Dice Movement, Drawing, Elapsed Real Time Ending, Enclosure, End Game Bonuses, Events,
Finale Ending, Flicking, Follow, Force Commitment, Grid Coverage, Grid Movement, Hand Management,
Hexagon Grid, Hidden Movement, Hidden Roles, Hidden Victory Points, Highest-Lowest Scoring, Hot Potato,
"I Cut, You Choose", Impulse Movement, Income, Increase Value of Unchosen Resources, Induction,
Interrupts, Investment, Kill Steal, King of the Hill, Ladder Climbing, Layering,
Legacy Game, Line Drawing, Line of Sight, Loans, Lose a Turn, Mancala,
Map Addition, Map Deformation, Map Reduction, Market, Matching, Measurement Movement,
Melding and Splaying, Memory, Minimap Resolution, Modular Board, Move Through Deck,
Movement Points, Movement Template, Moving Multiple Units, Multi-Use Cards, Multiple Maps,
Narrative Choice / Paragraph, Negotiation, Neighbor Scope, Network and Route Building,
Once-Per-Game Abilities, Open Drafting, Order Counters, Ordering, Ownership, Paper-and-Pencil,
Passed Action Token, Pattern Building, Pattern Movement, Pattern Recognition, Physical Removal,
Pick-up and Deliver, Pieces as Map, Player Elimination, Player Judge, Point to Point Movement,
Predictive Bid, Prisoner's Dilemma, Programmed Movement, Push Your Luck, Questions and Answers, Race,
Random Production, Ratio / Combat Results Table, Re-rolling and Locking, Real-Time, Relative Movement,
Resource Queue, Resource to Move, Rock-Paper-Scissors, Role Playing, Roles with Asymmetric Information,
Roll / Spin and Move, Rondel, Scenario / Mission / Campaign Game, Score-and-Reset Game, Secret Unit Deployment,
Selection Order Bid, Semi-Cooperative Game, Set Collection, Simulation, Simultaneous Action Selection,
Singing, Single Loser Game, Slide / Push, Solo / Solitaire Game, Speed Matching, Spelling, Square Grid,
Stacking and Balancing, Stat Check Resolution, Static Capture, Stock Holding, Storytelling, Sudden Death Ending,
Tags, Take That, Targeted Clues, Team-Based Game, Tech Trees / Tech Tracks, Three Dimensional Movement,
Tile Placement, Track Movement, Trading, Traitor Game, Trick-taking, Tug of War, Turn Order: Auction,
Turn Order: Claim Action, Turn Order: Pass Order, Turn Order: Progressive, Turn Order: Random,
Turn Order: Role Order, Turn Order: Stat-Based, Turn Order: Time Track, Variable Phase Order,
Variable Player Powers, Variable Set-up, Victory Points as a Resource, Voting, Worker Placement,
Worker Placement with Dice Workers, "Worker Placement, Different Worker Types", Zone of Control]


**Final Output**
mechanics: [A, B, ..., Z]
"""


### Complexity rating

In [ ]:
prompts['complexity'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Analyze Learning Complexity
- Analyze the length of the text, setup steps, rule exceptions and other factor that may indicate how rule intensive the game is.
- Reason about how quickly a new player could grasp the basics.
2. Analyze Playing Complexity
- Look at in‑game actions per turn, resource management, simultaneous moves and number of element to manage
- Estimate mental load during a typical play session.
3. Analyze Strategy/Tactics
- Examine depth of decision space, long‑term planning, branching possibilities, and how impactful is a wrong decision.
4. Convert each qualitative assessment to a numeric rating (1‑5)
- Provide a short justification for each number.
5 Compute the complexity rating
- Average = (Learning + Playing + Strategy) / 3
- Round to one decimal.

**Final Output**
Learning: X
Playing: Y
Strategy: Z
Overall Complexity: W.W"""

### Optimal player count

In [ ]:
prompts['player'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Identify the player‑count range stated in the rules (minimum‑maximum). If the rulebook does not explicitly state a player count, infer the appropriate range from the game components and mechanics described in the box contents.
2. Examine how the core mechanics scale with player number
3. Consider the impact on play time, player interaction, and variance (e.g., games that become chaotic with many players or too slow with few).
4. Weigh the pros and cons of each possible player count within the allowed range.
5. find the optimal player count based on your observations

**Final Output**
Optimal player count: X
"""

### Game duration

In [ ]:
prompts['duration'] = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Assess the progression of a typical game turn.
2. Evaluate the complexity of the required actions and how long a turn would last
3. Evaluate how much each turn brings the player closer to the final goal
4. Based on your observation estimate the game’s average duration in minutes.

**Final Output**
Average game duration: X minutes
"""

## Execution

In [ ]:
# select only prompt not executed
to_do = [g for g in FILE_NAMES if f'{CHOSEN}_{g}.json' not in os.listdir(BASE_FOLDER)
                                 or f'{CHOSEN}_{g}.json' in OVERWRITE]
if to_do == []:
    print('Nothing to do')

for g in to_do:
    print(g)
    output_dict = {p: {it: '' for it in range(IT)} for p in prompts.keys()}
    for p,it in tqdm([(p,it) for p in prompts.keys() for it in range(IT)]):
        for retry in range(MAX_RETRY):
            rulebook = requests.get(BASE_URL +g+'.txt').text
            name = g.replace('_',' ')
            out = model.create_chat_completion(generate_message(prompts[p], f'Here is the full rulebook of the game {name}:\n'+rulebook),
                                               temperature=models[CHOSEN]['temperature'],
                                               response_format = all_rf if p == 'all' else None,
                                               )['choices'][0]['message']['content']
            if(check_output(p,out)):
                break
        output_dict[p][it] = out
        if(VERBOSE):
            print(p,'-',it)
            print(out)

    with open(f'{BASE_FOLDER}{CHOSEN}_{g}.json','w') as f:
        json.dump(dict(output_dict),f)

ticket_to_ride


  2%|▏         | 1/50 [00:05<04:36,  5.64s/it]

all - 0
{ "reasoning": "Based on the provided rulebook, I will analyze the game mechanics, complexity, optimal player count, and estimated game duration of Ticket to Ride." }
  													


  4%|▍         | 2/50 [00:08<03:21,  4.20s/it]

all - 1
{"reasoning": "Analyzing the Ticket to Ride game rules to identify the key actions and components, map them to existing BGG mechanics, judge the rule density and decision depth, and estimate the game's average duration."}
																	


...

mechanics - 2
**Key Actions and Components**

1. Drawing cards from the deck and hand management
2. Playing Action cards to perform various actions (e.g., gaining cards, trashing cards, drawing cards, etc.)
3. Buying cards from the Supply using Treasure cards
4. Discarding cards from the hand and play area
5. Shuffling the deck when necessary
6. Using various Kingdom cards with unique abilities (e.g., Artisan, Bandit, Bureaucrat, etc.)
7. Counting Victory points at the end of the game

**Mapped to BGG Mechanics**

1. Drawing and Hand Management: **Hand Management**
2. Playing Action cards: **Action Queue**, **Action Points**, and **Action / Event**
3. Buying cards: **Action Points**, **Action Queue**, and **Action / Event**
4. Discarding cards: **Hand Management**
5. Shuffling the deck: **Automatic Resource Growth**
6. Using Kingdom cards: **Variable Player Powers** and **Card Play Conflict Resolution**
7. Counting Victory points: **End Game Bonuses** and **Victory Points as a Resource

 28%|██▊       | 14/50 [03:23<11:12, 18.69s/it]

mechanics - 3
**Key Actions and Components**

1. Drawing cards from the deck or discard pile
2. Playing cards from the hand to perform various actions
3. Trashing cards from the hand to gain new cards
4. Gaining cards from the Supply
5. Discarding cards to the discard pile
6. Buying cards from the Supply using Treasure cards
7. Revealing cards to other players
8. Setting aside cards for later use
9. Shuffling the deck or discard pile
10. Counting cards in the deck or discard pile

**Mapped to BGG Mechanics**

1. Drawing: Automatic Resource Growth, Deck Construction, Drawing
2. Playing cards: Action / Event, Action Queue, Action Retrieval, Simultaneous Action Selection
3. Trashing: Trash a card, Discard a card, Re-rolling and Locking
4. Gaining: Gain a card, Automatic Resource Growth, Deck Construction
5. Discarding: Discard a card, Trash a card
6. Buying: Auction / Bidding, Resource to Move, Trading
7. Revealing: Hidden Movement, Hidden Roles, Reveal a card
8. Setting aside: Set aside 

 30%|███       | 15/50 [03:44<11:24, 19.55s/it]

mechanics - 4
**Key Actions and Components**

1. Drawing cards
2. Discarding cards
3. Playing Action cards
4. Buying cards
5. Gaining cards
6. Trashing cards
7. Revealing cards
8. Looking at cards
9. Setting aside cards
10. Playing Treasure cards
11. Using +Actions, +Buys, and +<x> effects
12. Counting cards in the deck and discard pile
13. Shuffling the deck
14. Drawing a new hand of 5 cards
15. Ending the game

**Components**

1. Cards (Treasure, Victory, Curse, Action, Reaction)
2. Deck
3. Discard pile
4. Trash pile
5. Supply (piles of cards)
6. Play area (table space in front of each player)
7. Organizer tray
8. Rule booklet

**Mechanics**

Based on the rules and actions described, I identify the following mechanics:

1. **Deck-Building**: Players build their deck by drawing cards, discarding cards, and buying new cards.
2. **Card Play**: Players play Action cards to perform various actions, such as drawing cards, trashing cards, and gaining cards.
3. **Action Point Allowance Syste